In [1]:
import pandas as pd

DATA_PATH = "data/amazon_reviews_us_Digital_Video_Games_v1_00.tsv.zip"

df = pd.read_csv(DATA_PATH, sep="\t")
print("Default settings:", df.shape)

Default settings: (144724, 15)


Pandas loaded 144,724 rows and 15 columns.

In [2]:
df = pd.read_csv(DATA_PATH, sep="\t", quoting=3)  # 3 = treat quotes as plain text
print("quoting=3:", df.shape)

quoting=3: (145431, 15)


The default version loaded 144,724 rows and the quoting=3 version loaded 145,431, a difference of 707. Pandas did not warn me.

This shows code can run without errors and still be wrong, so you should always check your data, like the row count.

In [3]:
peek = df["review_body"].sample(5, random_state=2134)
for text in peek:
    print(text, "\n")

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable<br />the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.<br /><br />I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.<br /><br />So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They dont care once they ha

The reviews have HTML code mixed into the text.

It probably came from the reviews being collected from Amazon’s website, where text is stored as HTML, and the codes were never turned back into normal characters.

In [4]:
df["star_rating"].value_counts(normalize=True).sort_index()

star_rating
1    0.171697
2    0.053283
3    0.079962
4    0.140314
5    0.554744
Name: proportion, dtype: float64

The ratings are uneven. About 55% are 5 stars, 17% are 1 star, 14% are 4 stars, 8% are 3 stars, and only 5% are 2 stars. Most reviews are either very positive or very negative.

This matters because a model could look accurate just by guessing “positive” most of the time, since over half the reviews are 5 stars. Accuracy alone wouldn’t show how well it handles the less common ratings.

In [5]:
df["product_title"].value_counts().head(10)

product_title
Playstation Network Card                                          13642
Xbox Live Subscription                                             7307
Playstation Plus Subscription                                      4712
Xbox Live Gift Card                                                3438
SimCity - Limited Edition                                          3421
Xbox 360 Live Points Card                                          3036
Battlefield 4                                                      1453
Final Fantasy XIV: A Realm Reborn                                  1273
Battlefield 3                                                      1160
Command and Conquer The Ultimate Collection [Online Game Code]      875
Name: count, dtype: int64

Most of the top-reviewed “games” aren’t actually games. They’re gift cards and subscriptions, like the Playstation Network Card (13,642 reviews) and Xbox Live Subscription (7,307). Only a few, like SimCity and Battlefield, are real games. So many reviews are about whether a code worked, not about gameplay.

In [6]:
print("Reviews with <br />:", df["review_body"].str.contains("<br />").sum())
df = df.dropna(subset=["review_body"])
df["review_body"] = df["review_body"].str.replace("<br />", " ")
print("After cleaning:", df["review_body"].str.contains("<br />").sum(), "\n")

for text in df.loc[peek.index, "review_body"]:
    print(text, "\n")

Reviews with <br />: 25699
After cleaning: 0 

The &#34;AI&#34; is wrong by cheating, many bugs and control system cheating is not reliable the PC version is penalized 

Is fast secure and very easy you only payed and in less than a minute you can use the code thanks 

This is exactly what it says it is, and I was able to email it as a gift!  Perfect.  Thanks! 

Since there is no way to return downloaded games or even remark on my problem, my only other options is via a review.  I have a steam account and am logged in.  I tried installing the game, and it gave me a generic &#34;Trouble Communicating With Gateway&#34; message.  Googling the message returns one web page in German.  Retried three times with the same result.  So...  Looks like I'm screwed. 

I bought this crapware online but was not able to install it on my tablet nor my android phone. The amazon android market app \\"seems\\" to crash when installing free or promotional apps. Amazon customer service is non existent. They 

Before cleaning, 25,699 reviews had line breaks. After cleaning, 0 did.

The reviews still have “ instead of quote marks and extra backslashes like \”. I don’t think this would change whether a model reads a review as positive or negative, because the important words like “perfect” and “crapware” are still there.

In [7]:
sample = df.groupby("star_rating").sample(n=100, random_state=42)
complaints = df[df["star_rating"] == 1].sample(n=20, random_state=42)
print(len(sample), len(complaints))

500 20


Over half the reviews are 5 stars and only 5% are 2 stars. If we took 500 random reviews, we’d get mostly 5-star ones and very few 2 or 3-star ones. Taking 100 of each makes the test fair, so we can see how the model does on every star rating.

In [8]:
from transformers import pipeline

classifier = pipeline(
    "sentiment-analysis",
    model="distilbert/distilbert-base-uncased-finetuned-sst-2-english",
)
print(sample["star_rating"].iloc[0], sample["review_body"].iloc[0][:200])
classifier(sample["review_body"].iloc[0])

Loading weights:   0%|          | 0/104 [00:00<?, ?it/s]

1 Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get 


[{'label': 'NEGATIVE', 'score': 0.9998071789741516}]

The result has two keys: label and score. The label is the model’s prediction (POSITIVE or NEGATIVE), and the score is how confident the model is in that label, from 0 to 1.

The customer gave 1 star and wrote a negative review. The model said NEGATIVE with a score of 0.9998 (about 99.98% confident), so the model agrees with the customer’s rating.

In [9]:
%%time
results = classifier(sample["review_body"].tolist(), truncation=True)
sample["label"] = [r["label"] for r in results]
sample["score"] = [r["score"] for r in results]
sample[["star_rating", "label", "score"]].head(10)

CPU times: user 739 ms, sys: 127 ms, total: 866 ms
Wall time: 3.69 s


,star_rating,label,score
40836,1,NEGATIVE,0.999807
85497,1,NEGATIVE,0.990978
124434,1,NEGATIVE,0.998919
118528,1,NEGATIVE,0.999704
13480,1,NEGATIVE,0.999466
47179,1,NEGATIVE,0.999234
130193,1,NEGATIVE,0.999613
126902,1,NEGATIVE,0.999815
58985,1,NEGATIVE,0.999799
22608,1,NEGATIVE,0.999736


Classifying 500 reviews took 6.53 seconds. At that rate, all 145,000 reviews would take about 6.5 × 290 ≈ 1,900 seconds/32 minutes.

truncation=True protects against reviews that are too long. The model can only read about 512 tokens at a time, so without truncation a very long review would cause an error. With it, the model cuts the review off and only reads the beginning.


In [10]:
def stars_to_sentiment(stars):
    """Convert a star rating into the sentiment the customer expressed."""
    if stars <= 2:
        return "NEGATIVE"
    if stars >= 4:
        return "POSITIVE"
    return "NEUTRAL"


sample["truth"] = sample["star_rating"].apply(stars_to_sentiment)
graded = sample[sample["truth"] != "NEUTRAL"]
print("Accuracy:", (graded["label"] == graded["truth"]).mean())

Accuracy: 0.8775


We leave out the 3-star reviews because they are neutral, but the model can only answer POSITIVE or NEGATIVE. It would be wrong on every 3-star review no matter what, which wouldn’t be a fair test.

The accuracy is 0.8775 (about 88%). That’s good, but I wouldn’t fully trust it in a product. It gets about 1 in 8 reviews wrong and can’t handle neutral reviews, so it’s better for a rough overview than for judging individual reviews.

In [11]:
pd.crosstab(sample["star_rating"], sample["label"])

label,NEGATIVE,POSITIVE
star_rating,,
1,96,4
2,95,5
3,66,34
4,23,77
5,17,83


The model gets 4-star reviews wrong most often (23 of 100 called NEGATIVE).

For 3-star reviews, it had to pick a side: 66 NEGATIVE and 34 POSITIVE, so it leans negative.

In [12]:
mistakes = graded[graded["label"] != graded["truth"]]
for review in mistakes.head(5).itertuples():
    print(review.star_rating, review.label, round(review.score, 2))
    print(review.review_body[:300], "\n")

1 POSITIVE 0.99
Don't get me wrong, the game concept is awesome. Graphics are great.  BUT  If you don't want to lose your progress when the Simcity Servers stop responding, don't buy this. There is no option to save locally just saved to their server so the smallest error will lose your work.  Overall it's a good g 

1 POSITIVE 1.0
Enjoys game 

1 POSITIVE 0.73
I wasted $7. im sure its a good game but i cant even get to the game part. i tried. a lot. 

1 POSITIVE 1.0
Dose not work right on PSVITA so was not happy but it is a fun game love twisted metal games. 

2 POSITIVE 0.87
I bought this game for my 8 year old daughter who likes the kingdom of keflings game on xbox. The graphics are very simple and it is a basic strategy/building style game. Unfortunately, it looks like it would have been cutting edge 15 to 20 years ago, but it really didn't entertain my kids for more  



SimCity: It’s a mixed review that ends with “overall it’s a good game,” so the rating is misleading.
“Enjoys game”: The text is positive, so the customer probably picked the wrong rating.
“I wasted $7”: The customer is upset, but “good game” confused the model. Model’s fault.
PS Vita: It says the game doesn’t work but is fun, so it’s mixed. Both.
Keflings: The customer is disappointed, but the model missed “unfortunately.” Model’s fault.

In [13]:
star_model = pipeline(
    "sentiment-analysis",
    model="nlptown/bert-base-multilingual-uncased-sentiment",
)
results = star_model(sample["review_body"].tolist(), truncation=True)
sample["pred_stars"] = [int(r["label"][0]) for r in results]  # "4 stars" -> 4

print("Exact:", (sample["pred_stars"] == sample["star_rating"]).mean())
off_by = (sample["pred_stars"] - sample["star_rating"]).abs()
print("Within one star:", (off_by <= 1).mean())

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Exact: 0.574
Within one star: 0.912


It gets the exact star rating 57.4% of the time, and is within one star 91.2% of the time.

For a game publisher’s dashboard, I would use the star model. It gives more detail, can handle neutral 3-star reviews, and is within one star over 90% of the time. The first model is more accurate, but only because it has just two choices.

In [14]:
zsc = pipeline("zero-shot-classification", model="facebook/bart-large-mnli")

labels = ["price", "bugs or crashes", "account or login", "gameplay", "download or install"]
results = zsc(complaints["review_body"].tolist(), candidate_labels=labels)
complaints["topic"] = [r["labels"][0] for r in results]
complaints["topic"].value_counts()

Loading weights:   0%|          | 0/515 [00:00<?, ?it/s]

topic
download or install    5
bugs or crashes        5
account or login       4
price                  3
gameplay               3
Name: count, dtype: int64

Customers complain most about download or install problems and bugs or crashes (5 each), followed by account or login (4). Price and gameplay had the fewest (3 each). So most complaints are about technical problems, not the game itself.

In [15]:
for review in complaints.head(3).itertuples():
    print(review.topic)
    print(review.review_body[:300], "\n")

price
Terrible, confusing interface. Looks like a cartoon. Didn't really like the long, drawn out stories in my player. Game play and graphics weren't that much improved $30 extra we had to pay (Now we get VC yea! not). 

download or install
I too installed and tried to play Cobi Treasure Island. My PC does meet the proper requirements so I do not know what the problem is. 

gameplay
If you liked \\"Heavy Rain\\", you'll like this.  However, if you'd like for your games to have a faster pace than 'the English Patient', you'll want to skip this title in favor of something more exciting, like \\"Parking Simulator 4\\". 



Price: Partly makes sense. It mentions paying $30 extra, but most of the review is about the interface, story, and graphics, so gameplay would also fit.
Download or install: Makes sense. The customer installed the game but can’t get it to run.
Gameplay: Makes sense. It’s a sarcastic complaint that the game is too slow and boring.

In [16]:
more_labels = labels + ["technical problems", "HR policy"]
new_results = zsc(complaints["review_body"].head(5).tolist(), candidate_labels=more_labels)
for old, new in zip(results[:5], new_results):
    print("Before:", old["labels"][0], round(old["scores"][0], 2))
    print("After: ", new["labels"][0], round(new["scores"][0], 2), "\n")

Before: price 0.66
After:  price 0.59 

Before: download or install 0.41
After:  download or install 0.3 

Before: gameplay 0.36
After:  gameplay 0.32 

Before: bugs or crashes 0.53
After:  technical problems 0.7 

Before: download or install 0.79
After:  download or install 0.45 



What happened: Most top labels stayed the same, but all the scores dropped. One review switched from “bugs or crashes” to “technical problems.”

Why: The scores must add up to 1, so more labels means each one gets a smaller share. The overlapping label took points from similar ones.

What it tells us: Pick clear labels that don’t overlap.